# Vesuvius Surface Detection — Final Inference (Winner-Aligned)

**4-model nnU-Net ensemble + 5-step post-processing** (Kaggle 1st-place recipe).

Ensemble:
```
P = 0.12 · M1_128  +  0.28 · M2_192ft  +  0.42 · M3_256ft  +  0.18 · M4_192scratch
mask = P[surface] > 0.23
```
then: small-component removal → 1-voxel plug → height-map patching → binary closing → fill_holes.

**Usage:** set `INPUT_FOLDER` to any folder of `.tif` volumes and run all cells.

## Configuration

In [ ]:
# ==========================================
# CHANGE THESE PATHS
# ==========================================
INPUT_FOLDER  = '/raid/home/vikram_govt/Dikshant/gautam/cv/data/test_images'
OUTPUT_FOLDER = '/raid/home/vikram_govt/Dikshant/gautam/cv/notebooks/output_final'

# Optional GT for metrics (set None to skip)
GT_FOLDER = None  # e.g. '/raid/home/vikram_govt/Dikshant/gautam/cv/data/train_labels'

# Ensemble threshold (winner used 0.20 – 0.26)
THRESHOLD = 0.23

# Post-processing toggles (all True for final submission)
DO_SMALL_COMPONENT_FILTER = True
DO_VOXEL_PLUG            = True
DO_HEIGHTMAP_PATCH       = True
DO_CLOSING               = True
DO_FILL_HOLES            = True
MIN_COMPONENT_VOXELS     = 20_000
CLOSING_RADIUS           = 3

## Setup & Model Loading

In [ ]:
import sys, os
PROJECT_DIR = '/raid/home/vikram_govt/Dikshant/gautam/cv'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

os.environ['nnUNet_raw']          = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_raw')
os.environ['nnUNet_preprocessed'] = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_preprocessed')
os.environ['nnUNet_results']      = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_results')
os.environ['nnUNet_USE_BLOSC2']   = '1'
os.environ['nnUNet_compile']      = 'false'

import numpy as np
import torch
import tifffile
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from glob import glob

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

tif_files = sorted(glob(os.path.join(INPUT_FOLDER, '*.tif')))
print(f'Found {len(tif_files)} volumes in {INPUT_FOLDER}')
for f in tif_files:
    print(f'  {os.path.basename(f)}')

In [ ]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

RESULTS = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_results/Dataset200_VesuviusSurface')

# --- Ensemble members (winner-aligned weights) ---
MODEL_SPECS = [
    # (display_name, subdir, ensemble_weight)
    ('M1_128_MPlans_4000ep',     'nnUNetTrainer_4000epochs__nnUNetResEncUNetMPlans__3d_fullres',             0.12),
    ('M2_192ft_250ep',           'nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans_patch192__3d_fullres',     0.28),
    ('M3_256ft_250ep',           'nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans_patch256__3d_fullres',     0.42),
    ('M4_192scratch_4000ep',     'nnUNetTrainer_4000epochs__nnUNetResEncUNetMPlans_patch192__3d_fullres',    0.18),
]

predictors = []
total_weight = 0.0
for name, subdir, weight in MODEL_SPECS:
    model_dir = os.path.join(RESULTS, subdir)
    ckpt = os.path.join(model_dir, 'fold_all', 'checkpoint_best.pth')
    if not os.path.exists(ckpt):
        print(f'  MISSING {name} — skipping')
        continue
    p = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
                       device=device, verbose=False)
    p.initialize_from_trained_model_folder(model_dir, use_folds=('all',),
                                          checkpoint_name='checkpoint_best.pth')
    predictors.append((name, p, weight))
    total_weight += weight
    print(f'  Loaded {name}  weight={weight}')

# Renormalize weights in case some models are missing
if total_weight > 0 and abs(total_weight - 1.0) > 1e-6:
    predictors = [(n, p, w / total_weight) for n, p, w in predictors]
    print(f'  Renormalized weights (sum was {total_weight:.3f})')

print(f'\nEnsemble: {len(predictors)} models')
for n, _, w in predictors:
    print(f'  {w:.3f}  {n}')

## Inference + Post-Processing

In [ ]:
def nnunet_predict(predictor, volume_np):
    img = volume_np.astype(np.float32)[np.newaxis]
    _, probs = predictor.predict_single_npy_array(
        img, {'spacing': [1.0, 1.0, 1.0]}, None, None, True
    )
    return probs  # (C, D, H, W)

def ensemble_predict(predictors, volume_np):
    probs_sum = None
    per_model = {}
    for name, pred, weight in predictors:
        print(f'    [{name}] running...')
        probs = nnunet_predict(pred, volume_np)
        per_model[name] = probs
        probs_sum = weight * probs if probs_sum is None else probs_sum + weight * probs
        torch.cuda.empty_cache()
    return probs_sum, per_model

In [ ]:
# Winner's 5-step post-processing
from src.postprocess import full_postprocess, remove_small_components, plug_voxel_holes, patch_all_sheets, binary_close, fill_cavities

def apply_postprocess(surface_prob, threshold=THRESHOLD):
    return full_postprocess(
        surface_prob,
        threshold=threshold,
        min_component_voxels=MIN_COMPONENT_VOXELS,
        do_plug=DO_VOXEL_PLUG,
        do_patch=DO_HEIGHTMAP_PATCH,
        do_close=DO_CLOSING,
        do_fill=DO_FILL_HOLES,
        closing_radius=CLOSING_RADIUS,
    )

def to_3class(surface_mask, air_prob, papyrus_prob):
    out = np.zeros_like(surface_mask, dtype=np.uint8)
    out[surface_mask > 0] = 1
    non_surf = ~(surface_mask > 0)
    out[non_surf & (papyrus_prob > air_prob)] = 2
    return out

print('Post-processing ready.')

## Visualization helpers

In [ ]:
SEG_CMAP = ListedColormap(['black', 'red', 'dodgerblue'])

def save_comparison(image, pred_3c, sample_id, output_dir, gt=None):
    slices = [0.25, 0.5, 0.75]
    cols = 2 + (1 if gt is not None else 0)
    fig, axes = plt.subplots(len(slices), cols, figsize=(4*cols, 4*len(slices)))
    for row, frac in enumerate(slices):
        z = int(frac * image.shape[0])
        c = 0
        axes[row, c].imshow(image[z], cmap='gray'); axes[row, c].set_ylabel(f'z={z}'); c += 1
        if gt is not None:
            axes[row, c].imshow(image[z], cmap='gray', alpha=0.5)
            axes[row, c].imshow(gt[z], cmap=SEG_CMAP, alpha=0.55, vmin=0, vmax=2); c += 1
        axes[row, c].imshow(image[z], cmap='gray', alpha=0.5)
        axes[row, c].imshow(pred_3c[z], cmap=SEG_CMAP, alpha=0.55, vmin=0, vmax=2)
        for k in range(cols):
            axes[row, k].set_xticks([]); axes[row, k].set_yticks([])
    titles = ['CT Image'] + (['Ground Truth'] if gt is not None else []) + ['Ensemble + PostProc']
    for k, t in enumerate(titles):
        axes[0, k].set_title(t, fontweight='bold')
    plt.suptitle(f'Volume {sample_id}', fontweight='bold', y=1.01)
    plt.tight_layout()
    out = os.path.join(output_dir, f'{sample_id}_comparison.png')
    plt.savefig(out, dpi=140, bbox_inches='tight'); plt.show()
    print(f'    saved {out}')

def save_surface_prob(image, ens_prob, sample_id, output_dir, gt=None):
    z = image.shape[0] // 2
    cols = 2 + (1 if gt is not None else 0)
    fig, axes = plt.subplots(1, cols, figsize=(5*cols, 5))
    c = 0
    axes[c].imshow(image[z], cmap='gray'); axes[c].set_title('CT (mid-slice)'); c += 1
    if gt is not None:
        axes[c].imshow(image[z], cmap='gray', alpha=0.5)
        axes[c].imshow(gt[z] == 1, cmap='Reds', alpha=0.7); axes[c].set_title('GT Surface'); c += 1
    im = axes[c].imshow(ens_prob[z], cmap='hot', vmin=0, vmax=1)
    axes[c].set_title('Ensemble P(surface)')
    plt.colorbar(im, ax=axes[c], fraction=0.046)
    for k in range(cols):
        axes[k].set_xticks([]); axes[k].set_yticks([])
    plt.suptitle(f'Surface Probability — {sample_id}', fontweight='bold')
    plt.tight_layout()
    out = os.path.join(output_dir, f'{sample_id}_surface_prob.png')
    plt.savefig(out, dpi=140, bbox_inches='tight'); plt.show()
    print(f'    saved {out}')

## Run inference on all volumes

In [ ]:
for tif_path in tif_files:
    sample_id = Path(tif_path).stem
    print(f'\n{"="*60}\n  {sample_id}\n{"="*60}')

    image = tifffile.imread(tif_path)
    print(f'  volume {image.shape} {image.dtype}')

    gt = None
    if GT_FOLDER:
        gpath = os.path.join(GT_FOLDER, f'{sample_id}.tif')
        if os.path.exists(gpath):
            gt = tifffile.imread(gpath)
            print(f'  GT loaded')

    # --- Ensemble inference ---
    print('  ensembling:')
    probs_ens, _ = ensemble_predict(predictors, image)
    # probs_ens: (C, D, H, W) with C=3
    surface_prob = probs_ens[1]

    # --- Post-processing ---
    print(f'  post-processing (threshold={THRESHOLD})...')
    surface_mask = apply_postprocess(surface_prob, threshold=THRESHOLD)
    pred_3c = to_3class(surface_mask, probs_ens[0], probs_ens[2])
    print(f'    surface voxels: {(pred_3c == 1).sum():,}')

    # --- Save mask ---
    tif_out = os.path.join(OUTPUT_FOLDER, f'{sample_id}.tif')
    tifffile.imwrite(tif_out, pred_3c)
    print(f'  saved mask: {tif_out}')

    # --- Metrics if GT available ---
    if gt is not None:
        from src.training.metrics import SegmentationMetrics
        m = SegmentationMetrics(3, ['air','surface','papyrus'])
        m.update(torch.from_numpy(pred_3c), torch.from_numpy(gt))
        r = m.compute()
        print(f'  surface_dice = {r["surface_dice"]:.4f} | mean_dice = {r["mean_dice"]:.4f}')

    # --- Visualizations ---
    save_comparison(image, pred_3c, sample_id, OUTPUT_FOLDER, gt=gt)
    save_surface_prob(image, surface_prob, sample_id, OUTPUT_FOLDER, gt=gt)

    torch.cuda.empty_cache()

print(f'\nDone. Results in {OUTPUT_FOLDER}')

## (Optional) Threshold sweep

Re-run on a single volume to see dice vs threshold (needs GT).

In [ ]:
# SWEEP_ID = '...'  # uncomment and fill in a sample_id from training data
# if GT_FOLDER and SWEEP_ID:
#     from src.training.metrics import SegmentationMetrics
#     image = tifffile.imread(os.path.join(INPUT_FOLDER, f'{SWEEP_ID}.tif'))
#     gt = tifffile.imread(os.path.join(GT_FOLDER, f'{SWEEP_ID}.tif'))
#     probs_ens, _ = ensemble_predict(predictors, image)
#     surface_prob = probs_ens[1]
#     sweep = []
#     for t in [0.18, 0.20, 0.22, 0.24, 0.26, 0.28, 0.30, 0.35, 0.40]:
#         mask = apply_postprocess(surface_prob, threshold=t)
#         pred = to_3class(mask, probs_ens[0], probs_ens[2])
#         m = SegmentationMetrics(3, ['air','surface','papyrus'])
#         m.update(torch.from_numpy(pred), torch.from_numpy(gt))
#         r = m.compute()
#         sweep.append((t, r['surface_dice']))
#         print(f'  t={t:.2f}  surface_dice={r["surface_dice"]:.4f}')
#     ts, ds = zip(*sweep)
#     plt.figure(figsize=(7,4)); plt.plot(ts, ds, 'o-'); plt.xlabel('threshold'); plt.ylabel('surface dice')
#     plt.title(f'Threshold sweep — {SWEEP_ID}'); plt.grid(True)
#     plt.savefig(os.path.join(OUTPUT_FOLDER, f'{SWEEP_ID}_threshold_sweep.png'), dpi=140)
#     plt.show()